# RAG in a notebook with Docling for IBM watsonx and watsonx.ai

This notebook builds a complete retrieval-augmented generation (RAG) loop with no infrastructure to run:

| Step | Runs where | What it does |
|---|---|---|
| Convert | Docling for IBM watsonx | Layout analysis, OCR and table structure. No models or GPU on your machine |
| Chunk | This notebook | Splits each document along its sections, sized for the embedding model |
| Embed | watsonx.ai | `ibm/granite-embedding-278m-multilingual`, 768 dimensions |
| Store and retrieve | This notebook | A NumPy matrix and a cosine similarity |
| Answer | watsonx.ai | `ibm/granite-4-h-small`, prompted with the retrieved chunks |

It accompanies the [RAG in a Notebook](https://developer.dcls.saas.ibm.com/tutorials/notebook-rag) tutorial of the [Docling for IBM watsonx documentation](https://developer.dcls.saas.ibm.com/).

**Credentials.** Set these in the repository root `.env` (copy `.env.example`), in a `.env` next to this notebook, or as environment variables before starting Jupyter:
`DOCLING_SERVICE_URL`, `DOCLING_SERVICE_API_KEY`, `WATSONX_URL`, `WATSONX_APIKEY`, `WATSONX_PROJECT_ID`.

## Step 1: Install the packages

Skip this cell if you started Jupyter with `uv run --group notebook jupyter lab`, which installs everything from `pyproject.toml`.

In [ ]:
# Install the packages only if they are missing (for example in VS Code or watsonx.ai Studio).
import importlib.util

if importlib.util.find_spec("docling") is None:
    %pip install -q docling-client "docling-core[chunking]" ibm-watsonx-ai numpy python-dotenv


Load the credentials from `.env` (this folder or the repository root).

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

for env_file in (Path(".env"), Path("..") / ".env", Path("../..") / ".env"):
    if env_file.is_file():
        load_dotenv(env_file)
        print(f"Loaded {env_file}")
        break

for name in ["DOCLING_SERVICE_URL", "DOCLING_SERVICE_API_KEY", "WATSONX_URL", "WATSONX_APIKEY", "WATSONX_PROJECT_ID"]:
    print(f"{name:25s} {'set' if os.environ.get(name) else 'MISSING'}")

## Step 2: Convert the documents

Sources can be URLs or local file paths. This notebook uses two public papers: the Docling technical report and the DocLayNet paper.

In [ ]:
from docling.service_client import DoclingServiceClient

SOURCES = [
    "https://arxiv.org/pdf/2408.09869",  # Docling technical report
    "https://arxiv.org/pdf/2206.01062",  # DocLayNet
]

client = DoclingServiceClient(
    url=os.environ["DOCLING_SERVICE_URL"],
    api_key=os.environ["DOCLING_SERVICE_API_KEY"],
)

documents = {}
for source in SOURCES:
    result = client.convert(source=source)
    documents[source] = result.document
    print(f"{result.status.value:10s} {source}")

## Step 3: Chunk the documents

The hybrid chunker splits each document along its structure: a chunk never mixes two sections, and tables and lists stay whole when they fit. It then merges or splits pieces so that each chunk fits the token budget.

Use the **tokenizer of the embedding model** and its input limit, 512 tokens for Granite Embedding 278M. The first run downloads the tokenizer files (a few MB) from Hugging Face; no model weights are downloaded.

In [ ]:
from docling_core.transforms.chunker.hybrid_chunker import HybridChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer

tokenizer = HuggingFaceTokenizer.from_pretrained(
    model_name="ibm-granite/granite-embedding-278m-multilingual",
    max_tokens=512,
)
chunker = HybridChunker(tokenizer=tokenizer)

chunks = []
for source, doc in documents.items():
    for chunk in chunker.chunk(dl_doc=doc):
        pages = sorted({prov.page_no for item in chunk.meta.doc_items for prov in item.prov})
        chunks.append({
            "source": source,
            "headings": chunk.meta.headings or [],
            "pages": pages,
            # The contextualized text starts with the chunk's headings.
            # It embeds better than the bare chunk text.
            "text": chunker.contextualize(chunk=chunk),
        })

print(f"{len(chunks)} chunks")
print(chunks[3]["headings"], chunks[3]["pages"])
print(chunks[3]["text"][:300])

## Step 4: Embed and index

Embed every chunk with watsonx.ai and keep the vectors in a NumPy matrix. Normalizing them makes the dot product a cosine similarity.

In [ ]:
import numpy as np
from ibm_watsonx_ai import Credentials
from ibm_watsonx_ai.foundation_models import Embeddings
from ibm_watsonx_ai.metanames import EmbedTextParamsMetaNames

credentials = Credentials(
    url=os.environ["WATSONX_URL"],
    api_key=os.environ["WATSONX_APIKEY"],
)
project_id = os.environ["WATSONX_PROJECT_ID"]

embedder = Embeddings(
    model_id="ibm/granite-embedding-278m-multilingual",
    credentials=credentials,
    project_id=project_id,
    # Docling and watsonx.ai count tokens slightly differently: truncate
    # instead of failing on a chunk at the limit.
    params={EmbedTextParamsMetaNames.TRUNCATE_INPUT_TOKENS: 512},
)

vectors = np.array(embedder.embed_documents([c["text"] for c in chunks]))
vectors /= np.linalg.norm(vectors, axis=1, keepdims=True)
print(vectors.shape)  # (number of chunks, 768)


def retrieve(question: str, k: int = 5) -> list[dict]:
    """Return the k chunks closest to the question."""
    query = np.array(embedder.embed_query(question))
    query /= np.linalg.norm(query)
    scores = vectors @ query
    return [chunks[i] for i in np.argsort(-scores)[:k]]


for hit in retrieve("How many pages does DocLayNet contain?", k=3):
    print(hit["headings"], hit["pages"], hit["text"][:120].replace("\n", " "))

## Step 5: Answer questions with watsonx.ai

Number the retrieved chunks, give them to the chat model as context, and ask it to cite the numbers. Then print where each cited passage came from.

In [ ]:
from ibm_watsonx_ai.foundation_models import ModelInference

llm = ModelInference(
    model_id="ibm/granite-4-h-small",
    credentials=credentials,
    project_id=project_id,
)

SYSTEM_PROMPT = (
    "You answer questions using only the numbered passages provided. "
    "Cite the passages you use as [1], [2], ... "
    "If the passages do not contain the answer, say that you cannot answer from the documents."
)


def ask(question: str, k: int = 5) -> str:
    hits = retrieve(question, k=k)
    context = "\n\n".join(f"[{i}] {hit['text']}" for i, hit in enumerate(hits, start=1))
    response = llm.chat(
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Passages:\n{context}\n\nQuestion: {question}"},
        ],
        params={"temperature": 0, "max_tokens": 512},
    )
    answer = response["choices"][0]["message"]["content"]

    print(f"Q: {question}\n")
    print(f"A: {answer}\n")
    for i, hit in enumerate(hits, start=1):
        section = " > ".join(hit["headings"]) or "(no heading)"
        print(f"  [{i}] {hit['source']}, {section}, pages {hit['pages']}")
    return answer

In [ ]:
ask("How many pages does DocLayNet contain, and how were they annotated?");

In [ ]:
ask("Which AI models does Docling use for layout analysis and table structure?");

A question the documents do not cover: the model should say it cannot answer.

In [ ]:
ask("What is the capital of France?");

## Next steps

- The index lives in memory and is rebuilt every time you run the notebook. To keep it, store the same `text`, metadata and vectors in a vector database, for example [Milvus Lite](https://milvus.io/docs/milvus_lite.md).
- To index a whole document library, let the service chunk and deliver the chunks to OpenSearch or AstraDB: see [`03-vector-db-ingestion`](../../03-vector-db-ingestion).